# Lung nodule detection: training and scan-level evaluation on Kaggle

This notebook trains a YOLO detector on LUNA16 and evaluates it **per scan**, the way the
LUNA16 challenge does, using the [lung-nodule-detection](https://github.com/jonahernan99/lung-nodule-detection)
package.

**Before running:**
1. *Settings → Accelerator*: choose a GPU. *Settings → Internet*: on.
2. *Add Input*: the dataset **vafaeii/luna16** ("LUNA16 (v3)"), which contains all ten subsets,
   `annotations.csv` and `annotations_excluded.csv`.
3. Leave `QUICK_RUN = True` for the first run (a few minutes). Set it to `False` for a full run.
4. Choose the input with `INPUT_MODE`: `"gray"` uses one slice (the baseline), `"stack"` puts the
   slices 2 mm below and above in the other color channels (2.5D), aimed at small nodules.

In [ ]:
!pip install -q "lung-nodule-detection[train] @ git+https://github.com/jonahernan99/lung-nodule-detection.git"

## Settings

In [ ]:
from pathlib import Path

QUICK_RUN = True  # True: small smoke test. False: full training and evaluation.
INPUT_MODE = "gray"  # "gray": one slice (baseline). "stack": neighbouring slices as channels (2.5D).
STACK_MM = 2.0  # distance to each neighbouring slice in "stack" mode
PATIENCE = 20  # stop when validation fitness has not improved for this many epochs

MAX_SCANS = 20 if QUICK_RUN else None  # scans per split
EPOCHS = 10 if QUICK_RUN else 100
TEST_SUBSET, VAL_SUBSET = 9, 8
WORK = Path("/kaggle/working")
RUN_NAME = f"nodules-{INPUT_MODE}"

## Locate the LUNA16 files

In [ ]:
INPUT = Path("/kaggle/input")
# The dataset also ships a copy of annotations.csv inside evaluationScript/; take the top-level one.
ANNOTATIONS = min(INPUT.rglob("annotations.csv"), key=lambda p: len(p.parts))
LUNA_ROOT = ANNOTATIONS.parent
EXCLUDED = next(INPUT.rglob("annotations_excluded.csv"), None)
print("LUNA16 root:  ", LUNA_ROOT)
print("Annotations:  ", ANNOTATIONS)
print("Excluded:     ", EXCLUDED)

## Build the training data

Scans are split by LUNA16 subset, so every slice of a scan stays in one split. Test scans are
not exported as slices: they are evaluated on full volumes below. The input mode is saved in
`dataset/preprocessing.json`, and the evaluation reads it back, so training and evaluation always
see the same kind of image.

In [ ]:
from lungnodule.dataset import build_dataset
from lungnodule.preprocess import InputConfig

summary = build_dataset(
    LUNA_ROOT,
    ANNOTATIONS,
    WORK / "dataset",
    test_subset=TEST_SUBSET,
    val_subset=VAL_SUBSET,
    max_scans_per_split=MAX_SCANS,
    input_config=InputConfig(mode=INPUT_MODE, stack_mm=STACK_MM),
    progress=lambda uid, split, n: print(f"{split:<5} {n:>4} slices  {uid}"),
)
print(summary)

## Train

In [ ]:
from ultralytics import YOLO, settings

# Kaggle preinstalls Ray; without this, Ultralytics calls its tuning callback and prints warnings.
settings.update({"raytune": False})

model = YOLO("yolov5su.pt")  # YOLOv5s backbone with Ultralytics' anchor-free head
# In the first full run, validation loss rose after about epoch 35 (see the README). Early
# stopping ends training PATIENCE epochs after the best validation fitness, and best.pt, the
# model evaluated below, keeps that best epoch.
# In "stack" mode the color channels are different slices, so hue and saturation augmentation,
# which mix channels, are turned off. Brightness augmentation scales all channels alike and stays.
color = {"hsv_h": 0.0, "hsv_s": 0.0} if INPUT_MODE == "stack" else {}
model.train(
    data=str(WORK / "dataset" / "data.yaml"),
    epochs=EPOCHS,
    imgsz=512,
    batch=16,
    seed=0,
    patience=PATIENCE,
    project=str(WORK / "runs"),
    name=RUN_NAME,
    exist_ok=True,
    **color,
)
WEIGHTS = WORK / "runs" / RUN_NAME / "weights" / "best.pt"
print("Weights:", WEIGHTS)

## Evaluate per scan

The detector runs on every slice of each test scan, detections are merged into 3D candidates,
and FROC/CPM is computed with false positives counted **per scan**, ignoring candidates on
LUNA16's excluded findings.

In [ ]:
import json

from lungnodule.dataset import load_input_config
from lungnodule.evaluation import evaluate_split
from lungnodule.inference import ultralytics_detector

input_config = load_input_config(WORK / "dataset")
print("Input:", input_config)
detector = ultralytics_detector(str(WEIGHTS), confidence=0.01, image_size=512)
result = evaluate_split(
    LUNA_ROOT,
    ANNOTATIONS,
    WORK / "dataset" / "splits.csv",
    detector,
    WORK / "results",
    excluded_csv=EXCLUDED,
    max_scans=MAX_SCANS,
    input_config=input_config,
    progress=lambda uid, d, c: print(f"{d:>5} detections -> {c:>4} candidates  {uid}"),
)
print(f"Scans: {result.total_scans}, nodules: {result.total_nodules}")
print(
    f"Found nodules: {result.found_nodules}, false positives: {result.false_positives}, "
    f"duplicates ignored: {result.duplicate_candidates}, on excluded: {result.excluded_candidates}"
)
for rate, sensitivity in zip(result.fp_rates, result.sensitivities, strict=True):
    print(f"{rate:>6g} FP/scan  sensitivity {sensitivity:.3f}")
interval = json.loads((WORK / "results" / "froc.json").read_text())["cpm_interval"]
print(f"CPM: {result.cpm:.4f}  (95% CI {interval['lower']:.3f}-{interval['upper']:.3f})")

## FROC curve

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(result.curve_fps_per_scan, result.curve_sensitivity, drawstyle="steps-post")
ax.scatter(result.fp_rates, result.sensitivities, zorder=3)
ax.set_xscale("log", base=2)
ax.set_xlim(0.1, 10)
ax.set_ylim(0, 1)
ax.set_xlabel("Average false positives per scan")
ax.set_ylabel("Sensitivity")
ax.set_title(f"Scan-level FROC on {result.total_scans} test scans (CPM {result.cpm:.3f})")
ax.grid(True, which="both", alpha=0.3)
fig.tight_layout()
fig.savefig(WORK / "results" / "froc.png", dpi=150)
plt.show()

## Download the results

Everything needed for the README is zipped into one file. From the **Output** panel of the saved
version, download `lung_nodule_results_<input mode>.zip` (use the download button, not the browser's print
dialog, so the files keep full precision). It contains:
- `results/froc.json`: CPM with its confidence interval, sensitivities and the full curve
- `results/candidates.csv`: detections in the LUNA16 submission format
- `results/froc.png`: the FROC curve
- `runs/nodules-<input mode>/results.png` and `results.csv`: training curves and per-epoch metrics
- `dataset/preprocessing.json`: the input representation

A quick run uses few scans, so its numbers only show that the pipeline works. Report results
from a full run.

In [ ]:
import zipfile

files = [
    "results/froc.json",
    "results/candidates.csv",
    "results/froc.png",
    f"runs/{RUN_NAME}/results.png",
    f"runs/{RUN_NAME}/results.csv",
    f"runs/{RUN_NAME}/args.yaml",
    "dataset/preprocessing.json",
]
archive_path = WORK / f"lung_nodule_results_{INPUT_MODE}.zip"
with zipfile.ZipFile(archive_path, "w", zipfile.ZIP_DEFLATED) as archive:
    for name in files:
        if (WORK / name).exists():
            archive.write(WORK / name, name)
            print("added", name)
print("Download:", archive_path)